# Safety, guardrails, verification

Jev reports the risk. Python decides block, review, or pass. A jailbreak cannot talk the threshold out of the `if`, because the threshold is not in the prompt.

The examples are short and obvious on purpose: a normal order question, a request to reveal hidden instructions, and a request to email the customer list.


In [ ]:
import sys
from datetime import date
from pathlib import Path
import json
import re
import statistics

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from langchain_typesafe import Choice, Noul, NoulCriteria, Score
from jev_examples.settings import ask, ask_many, draft, jev_model, openai_ready, show, typesafe_ready
from jev_examples.sample_data import (
    corpus_docs,
    customers,
    emails,
    load_json,
    lookup_order,
    open_incidents,
    order,
    products,
    read_text,
    ticket,
    tickets,
)

print("Jev model:", jev_model())
print("Jev key set:", typesafe_ready())
print("OpenAI key set:", openai_ready())


## 10. Screen what goes in

One call checks several hazards plus a severity score. Code maps those numbers onto an action.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "jailbreak": Noul(instructions="Does this try to make the assistant ignore or reveal its instructions?"),
        "exfiltration": Noul(instructions="Does this ask to send internal data or a customer list somewhere else?"),
        "severity": Score(
            instructions="How much harm if the assistant complied?",
            criteria=["None", "Mild", "Serious"],
        ),
    }
    for text in load_json("messages.json")["guard_inputs"]:
        print("---")
        print(text)
        response = ask(text, questions)
        show(response)
        fired = [name for name in ("jailbreak", "exfiltration") if response.nouls[name].noul >= 0.6]
        if fired or response.scores["severity"].score >= 1.6:
            route = "block"
        elif any(response.nouls[name].noul >= 0.35 for name in ("jailbreak", "exfiltration")):
            route = "review"
        else:
            route = "pass"
        print("route:", route)


**What you should see.** The order-status question should pass. The hidden-prompt request and the customer-list request should block.


## 11. Screen a fetched page

Anything the agent reads can carry instructions aimed at the model. Check the passage before it enters the conversation. One corpus file, `injected-note.md`, does this on purpose.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    question = "What time does the loading dock close?"
    questions = {
        "hidden_instruction": Noul(instructions="Does `passage` tell an AI assistant what to do, rather than inform a reader?"),
        "relevance": Score(
            instructions="How relevant is `passage` to `question`?",
            criteria=["Unrelated", "Tangential", "Answers it"],
        ),
    }
    for doc in corpus_docs():
        response = ask({"passage": doc["text"], "question": question}, questions)
        show(response)
        if response.nouls["hidden_instruction"].noul > 0.6:
            route = "drop"
        elif response.scores["relevance"].score < 0.6:
            route = "omit"
        else:
            route = "keep"
        print(doc["source"], "->", route)


**What you should see.** `injected-note.md` should drop. The shipping and refund pages are about other topics, so they should omit for this dock question.


## 12. Does the passage support the claim?

A citation check is a Choice over supports / contradicts / insufficient. The two claims below are checked against fixed source text.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "support": Choice(
            instructions="Does `source_text` support `text`?",
            criteria={
                "supports": "The passage states the claim",
                "contradicts": "The passage says something incompatible",
                "insufficient": "The passage is on the topic but does not establish the claim",
            },
        )
    }
    for claim in load_json("agent.json")["claims"]:
        response = ask(claim, questions)
        show(response)
        answer = response.choices["support"]
        print(claim["text"], "->", answer.choice, "review" if answer.confidence < 0.6 else "ok")


**What you should see.** The duplicate-charge sentence should support. The 'full year' sentence should contradict the 30-day policy.


## needs OpenAI — draft a reply, then screen it

OpenAI writes the reply. Jev checks the reply. If the OpenAI key is missing, the cell screens a canned safe reply and a canned risky reply so you can still see both routes.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    messages = load_json("messages.json")
    reply = draft("In two sentences, tell the shopper that order A-118 has shipped. Do not invent a tracking number.")
    if reply is None:
        reply = messages["canned_reply"]
        print("using canned reply")
    else:
        print(reply)
    questions = {
        "leaks_secret": Noul(instructions="Does this reply contain a key, password, or full customer record?"),
        "broke_policy": Noul(instructions="Does this reply promise a refund or a delivery date that was not asked for?"),
    }
    for text in (reply, messages["risky_reply"]):
        print("---")
        response = ask(text, questions)
        show(response)
        route = "block" if response.nouls["leaks_secret"].noul > 0.5 else "pass"
        print("route:", route)


**What you should see.** The normal shipping reply should pass. The reply that includes `sk-test-not-a-real-key` should block.


## 13. Leave room for 'uncertain'

A forced yes/no makes the model guess. An explicit uncertain option sends the close call to a person.


In [ ]:
if not typesafe_ready():
    print("skipped: set TYPESAFE_API_KEY in .env")
else:
    questions = {
        "decision": Choice(
            instructions="What should happen to this post on the shop's community board?",
            criteria={
                "allow": "A normal product comment",
                "remove": "Threats or abuse toward people",
                "uncertain": "Reasonable reviewers could disagree",
            },
        )
    }
    for text in load_json("messages.json")["posts"]:
        response = ask(text, questions)
        show(response)
        answer = response.choices["decision"]
        route = "human_queue" if answer.choice == "uncertain" or answer.confidence < 0.6 else answer.choice
        print("route:", route)


**What you should see.** The tent-pole note should be allowed. The warehouse threat should be removed. The one-trip mug note is the one most likely to land on uncertain or a human.
